# EviChange — Geo-VLM server (Qwen2.5-VL on a Colab GPU)

Runs Qwen2.5-VL on Colab's GPU and makes it reachable from the web app on your computer, so
**🔎 Find changes → 🛰 Check with Geo-VLM** works. The app sends before/after image chips of a
change region; the model answers whether it sees real change inside the region's box.

**Steps**
1. `Runtime → Change runtime type` → **L4 GPU** (Colab Pro; loads the 7B model), then run the cells top to bottom.
   Step 1b asks for Google Drive access once per session: the model is cached there so later
   sessions don't download it again.
2. Step 5 prints two lines, `VLM_URL` and `VLM_TOKEN`. Set them on your computer (see the
   end of this notebook) and start the web app.
3. Keep this tab open while you use the check. When Colab stops, the URL stops working; rerun
   to get a new one.

**Security.** The tunnel URL is public, so every request must carry the random token printed
below; anything else gets `401`. Don't share the token. Stop the runtime when you're done.

**Which model.** `MODEL` in Step 2 picks it. `'auto'` (default) loads `Qwen2.5-VL-7B-Instruct` on GPUs
over 20 GB, else `Qwen2.5-VL-3B-Instruct` (same rule as `EviChange_Person2_starter.ipynb`).
`'qwen3-8b'` loads `Qwen3-VL-8B-Instruct`, for the model comparison before the Phase 4 freeze. To
switch, change `MODEL` and use `Runtime → Restart session`, then run all cells again (two 7–8B models
don't fit on an L4 together). The server reports which model it loaded, and the app shows it on
every result.

| Runtime | GPU memory | Model | Use for |
|---|---|---|---|
| **L4** (Colab Pro) | 24 GB | 7B | Default: Geo-VLM checks and Phase 4 prompt development |
| **A100** (Colab Pro) | 40 GB | 7B | The final frozen Phase 4 runs, if L4 is too slow or unavailable |
| T4 (free) | 15 GB | 3B | Quick tests only; never for reported results (the plan requires 7B) |

Colab Pro bills GPU time in compute units (roughly L4 ≈ 5 / h, A100 ≈ 12–15 / h; Colab's
**Resources** panel shows the exact rate). An idle server still counts and closing the tab doesn't
stop it, so run the **Stop** cell at the end when you're done. Keep the same model and GPU type for
all reported Phase 4 runs.

## Step 1 — Install

In [1]:
!pip -q install -U "transformers>=4.57" accelerate peft fastapi uvicorn pillow   # 4.57+: Qwen3-VL

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.


## Step 1b — Cache the model on Google Drive

The first session downloads the model into Drive (7B ≈ 17 GB, so Drive needs about 20 GB free);
later sessions load it from there. Without enough space it downloads to the runtime as before.
Run this **before** Step 2: the cache location is read when `transformers` is imported. If loading
from Drive turns out slower than a fresh download (Step 2 prints the load time), set
`USE_DRIVE_CACHE = False`.

In [2]:
import glob, os, shutil
USE_DRIVE_CACHE = False
CACHE_DIR = '/content/drive/MyDrive/evichange_hf_cache'

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    cached = glob.glob(f'{CACHE_DIR}/hub/models--Qwen--Qwen2.5-VL-*')
    free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
    if cached or free_gb > 20:
        os.environ['HF_HOME'] = CACHE_DIR
        print('Model cache on Drive:', CACHE_DIR, '(already has: ' + ', '.join(os.path.basename(c) for c in cached) + ')' if cached else '(first download, takes a few minutes)')
    else:
        print(f'Only {free_gb:.0f} GB free on Drive (need ~20 GB): downloading to the runtime instead')

## Step 2 — Load the model

`MODEL`: `'auto'`, `'qwen2.5-7b'`, `'qwen2.5-3b'` or `'qwen3-8b'`. Keep the same model for all reported runs.

`ADAPTER`: leave `''` for the base model, or the Drive path of a fine-tuned LoRA adapter (Phase 4b, D17),
e.g. `'/content/drive/MyDrive/evichange_ft/check'`. The server then reports the model as
`<model>+ft:<adapter folder>`, so its answers are logged apart from the base model's.

In [3]:
import os
import torch
from transformers import AutoModelForImageTextToText as VLM, AutoProcessor

MODEL = 'auto'
ADAPTER = ''   # e.g. '/content/drive/MyDrive/evichange_ft/check' (needs Drive mounted)
MODELS = {'qwen2.5-7b': 'Qwen/Qwen2.5-VL-7B-Instruct', 'qwen2.5-3b': 'Qwen/Qwen2.5-VL-3B-Instruct',
          'qwen3-8b': 'Qwen/Qwen3-VL-8B-Instruct'}

assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> L4 GPU'
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if MODEL == 'auto':
    MODEL = 'qwen2.5-7b' if gpu_gb > 20 else 'qwen2.5-3b'
MODEL_ID = MODELS[MODEL]
assert gpu_gb > 20 or MODEL == 'qwen2.5-3b', f'{MODEL_ID} needs a GPU over 20 GB (L4 or A100)'
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

import time
t0 = time.time()
model = VLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map='auto')
if MODEL_ID.startswith('Qwen/Qwen2.5'):
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=1024*28*28)
else:   # Qwen3-VL sizes images itself; the 512 px chips are well inside its default limits
    processor = AutoProcessor.from_pretrained(MODEL_ID)
if ADAPTER:
    from peft import PeftModel
    if not os.path.isdir(ADAPTER):
        from google.colab import drive
        drive.mount('/content/drive')
    model = PeftModel.from_pretrained(model, ADAPTER)
    MODEL_ID = f"{MODEL_ID}+ft:{os.path.basename(ADAPTER.rstrip('/'))}"
print('Loaded', MODEL_ID, f'on {gpu_gb:.0f} GB GPU in {time.time() - t0:.0f} s')
if '3B' in MODEL_ID:
    print('WARNING: 3B model (GPU under 20 GB). Fine for testing; use an L4 or A100 runtime for anything reported.')

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-VL-7B-Instruct on 24 GB GPU in 60 s


## Step 3 — Generation (same as the Person 2 notebook: greedy, so answers are repeatable)

In [4]:
import threading
gpu_lock = threading.Lock()   # one request on the GPU at a time

def generate(prompt, images=(), max_new_tokens=400):
    content = [{'type': 'image'} for _ in images] + [{'type': 'text', 'text': prompt}]
    messages = [{'role': 'user', 'content': content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    with gpu_lock:
        inputs = processor(text=[text], images=list(images) or None, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

print(generate('Say hello in five words.'))

Hello, how are you?


## Step 4 — Start the server (port 8001, token-protected)

In [5]:
import base64, io, secrets, time
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from PIL import Image

TOKEN = secrets.token_urlsafe(24)
api = FastAPI()

class GenerateRequest(BaseModel):
    prompt: str
    images: list[str] = []          # base64 PNGs
    max_new_tokens: int = 400

@api.get('/health')
def health():
    return {'model': MODEL_ID}

@api.post('/generate')
def generate_endpoint(req: GenerateRequest, authorization: str = Header(default='')):
    if not secrets.compare_digest(authorization, f'Bearer {TOKEN}'):
        raise HTTPException(401, 'bad token')
    if len(req.images) > 8 or req.max_new_tokens > 1500:
        raise HTTPException(400, 'too many images or tokens')
    imgs = [Image.open(io.BytesIO(base64.b64decode(b))).convert('RGB') for b in req.images]
    t0 = time.time()
    text = generate(req.prompt, imgs, req.max_new_tokens)
    print(f'answered in {time.time() - t0:.1f} s: {text[:80]!r}')
    return {'text': text, 'model': MODEL_ID}

server = uvicorn.Server(uvicorn.Config(api, host='127.0.0.1', port=8001, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print('server running on port 8001')

server running on port 8001


## Step 5 — Open the tunnel and print the settings for your computer

In [6]:
import os, re, subprocess, urllib.request
if not os.path.exists('/content/cloudflared'):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '/content/cloudflared')
    os.chmod('/content/cloudflared', 0o755)
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8001'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        URL = m.group(0); break
assert URL, 'tunnel did not start; rerun this cell'
time.sleep(5)   # give the tunnel a moment before the first request
print('Set these on your computer, then start the web app:\n')
print('  PowerShell:')
print(f'    $env:VLM_URL = "{URL}"')
print(f'    $env:VLM_TOKEN = "{TOKEN}"')
print('  Command Prompt (cmd):')
print(f'    set VLM_URL={URL}')
print(f'    set VLM_TOKEN={TOKEN}')

Set these on your computer, then start the web app:

  PowerShell:
    $env:VLM_URL = "https://preferred-brandon-honolulu-collectables.trycloudflare.com"
    $env:VLM_TOKEN = "FyreNLUQEmk1Fo38nd-7wicP7JNmyG4R"
  Command Prompt (cmd):
    set VLM_URL=https://preferred-brandon-honolulu-collectables.trycloudflare.com
    set VLM_TOKEN=FyreNLUQEmk1Fo38nd-7wicP7JNmyG4R


## On your computer

In the **same** terminal where you set `VLM_URL` and `VLM_TOKEN`, start the web app from the repo root:

    python -m uvicorn webapp.backend.main:app --port 8000

Open http://localhost:8000 → **🔎 Find changes**. The Geo-VLM section shows the model name when it's
connected. Each region takes roughly 20–40 s (Earth Engine renders the chips, then the model answers).

The 30 labelled evaluation sites are skipped until the Phase 4 prompts are frozen
(`agents/vlm_check.py`, `LABELLED_SITES_UNLOCKED`).

## Stop — when you're done

Releases the GPU so compute units stop draining (same as `Runtime → Disconnect and delete
runtime`). The tunnel URL stops working; the model stays cached on Drive.

In [ ]:
from google.colab import runtime
runtime.unassign()